In [0]:
# Instalar dependencias desde requirements.txt
%pip install -r ../requirements.txt
%restart_python

In [0]:
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")
dbutils.widgets.text("endpoint_name", "doc_vector_endpoint", "Endpoint Name")
dbutils.widgets.dropdown("endpoint_type", "STANDARD", ["STANDARD", "STORAGE_OPTIMIZED"], "Endpoint Type")
dbutils.widgets.text("source_table_name", "bluetab.rag.docs_text", "Source Table Name")
dbutils.widgets.text("index_name", "bluetab.rag.docs_text_idx", "Index Name")
dbutils.widgets.dropdown("pipeline_type", "TRIGGERED", ["TRIGGERED", "SCHEDULED"], "Pipeline Type")
dbutils.widgets.text("primary_key", "id", "Primary Key")
dbutils.widgets.text("embedding_source_column", "text", "Embedding Source Column")
dbutils.widgets.text("embedding_vector_column", "embedding", "Embedding Vector Column")
dbutils.widgets.text("embedding_model_endpoint_name", "simple_embbeidng", "Embedding Model Endpoint Name")
dbutils.widgets.text("embedding_dimension", "768", "Embedding Dimension")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# MLflow run management
dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
dbutils.widgets.text("current_run", "", "Current Run")

In [0]:
# Obtener valores de los widgets y definir variables principales
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
ENVIRONMENT = dbutils.widgets.get("environment")
endpoint_name = dbutils.widgets.get("endpoint_name")
endpoint_type = dbutils.widgets.get("endpoint_type")
source_table_name = dbutils.widgets.get("source_table_name")
index_name = dbutils.widgets.get("index_name")
pipeline_type = dbutils.widgets.get("pipeline_type")
primary_key = dbutils.widgets.get("primary_key")
embedding_source_column = dbutils.widgets.get("embedding_source_column")
embedding_vector_column = dbutils.widgets.get("embedding_vector_column")
embedding_model_endpoint_name = dbutils.widgets.get("embedding_model_endpoint_name")
embedding_dimension = dbutils.widgets.get("embedding_dimension")

# MLflow configuration
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# Variables globales para gestión de parent/child runs
PARENT_RUN_ID = dbutils.widgets.get("parent_run_id") or None
CURRENT_RUN = dbutils.widgets.get("current_run") or None

FULL_IDX_TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{index_name}"

print(f"Endpoint Name: {endpoint_name}")
print(f"Endpoint Type: {endpoint_type}")
print(f"Source Table Name: {source_table_name}")
print(f"Index Name: {index_name}")
print(f"Pipeline Type: {pipeline_type}")
print(f"Primary Key: {primary_key}")
print(f"Embedding Source Column: {embedding_source_column}")
print(f"Embedding Vector Column: {embedding_vector_column}")
print(f"Embedding Model Endpoint Name: {embedding_model_endpoint_name}")
print(f"Embedding Dimension: {embedding_dimension}")

In [0]:
%run "./00 Configuration and Utils"

In [0]:
start_child_run("08_create_index")

## 1. Create index endpoint

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

endpoints = client.list_endpoints()
endpoint_name = dbutils.widgets.get("endpoint_name")
endpoint_type = dbutils.widgets.get("endpoint_type")

if not any(endpoint['name'] == endpoint_name for endpoint in endpoints['endpoints']):
    client.create_endpoint(
        name=endpoint_name,
        endpoint_type=endpoint_type
    )
    print(f"Endpoint '{endpoint_name}' created.")
else:
    print(f"Endpoint '{endpoint_name}' already exists.")

## 2. Create index

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

endpoint_name = dbutils.widgets.get("endpoint_name")
source_table_name = dbutils.widgets.get("source_table_name")
index_name = dbutils.widgets.get("index_name")
pipeline_type = dbutils.widgets.get("pipeline_type")
primary_key = dbutils.widgets.get("primary_key")
embedding_source_column = dbutils.widgets.get("embedding_source_column")
embedding_model_endpoint_name = dbutils.widgets.get("embedding_model_endpoint_name")

# Obtener la lista de índices en el endpoint
indices = client.list_indexes(endpoint_name)

# Comprobar si existe
index_exists = any(idx['name'] == index_name for idx in indices['vector_indexes'])

if not index_exists:
    try:
        index = client.create_delta_sync_index(
          endpoint_name=endpoint_name,
          source_table_name=source_table_name,
          index_name=index_name,
          pipeline_type=pipeline_type,
          primary_key=primary_key,
          embedding_source_column=embedding_source_column,
          embedding_model_endpoint_name=embedding_model_endpoint_name,
          embedding_vector_column=embedding_vector_column,
          embedding_dimension=embedding_dimension
        )
        print(f"Index '{index_name}' created.")
    except Exception as e:
        print(f"Failed to create index '{index_name}': {e}")
else:
    print(f"Index '{index_name}' already exists..")